# Análise do Funil — AI Banking Sales Agent

Análise das métricas do funil omnicanal (URA + WhatsApp) para identificar gargalos e oportunidades.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/funnel_metrics.csv')
df['mes'] = pd.to_datetime(df['mes'])
print(df.to_string())

## 1. Funil em Cascata

In [ ]:
for _, row in df.iterrows():
    label = f"{row['mes'].strftime('%b/%Y')} {row['canal'].upper()}"
    etapas = {'Base': row['base_dia'], 'Atendimentos': row['atendimentos'],
              'Completadas': row['completadas'], 'Leads': row['leads'], 'Contas': row['contas_abertas']}
    print(f'\n{label}')
    for etapa, v in etapas.items():
        pct = v / row['base_dia'] * 100
        print(f'  {etapa:<14} {v:>8,.0f}  ({pct:.2f}%)')

## 2. Evolução das Taxas — URA

In [ ]:
ura = df[df['canal']=='ura'].sort_values('mes')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, titulo, cor) in zip(axes, [('taxa_atendimento','Atendimento','#2196F3'),('taxa_lead','Lead','#4CAF50'),('taxa_abertura','Conta Aberta','#FF9800')]):
    ax.bar(ura['mes'].dt.strftime('%b/%y'), ura[col]*100, color=cor, alpha=0.8)
    ax.set_title(titulo, fontweight='bold')
    for i,v in enumerate(ura[col]*100): ax.text(i, v+0.2, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3. ROI Estimado

In [ ]:
nov = df[(df['mes'].dt.month==11) & (df['canal']=='ura')].iloc[0]
CUSTO_MES = nov['atendimentos'] * 30 * 0.33
RECEITA_MES = nov['contas_abertas'] * 30 * 800
print(f'Custo IA/mes: R$ {CUSTO_MES:,.0f}')
print(f'Receita/mes:  R$ {RECEITA_MES:,.0f}')
print(f'ROI:          {RECEITA_MES/CUSTO_MES:.1f}x')

## 4. Gargalos Identificados

In [ ]:
gargalos = [('Base->Atendimento','~25%','Horario sub-otimo ou CNAE com baixa propensao','Testar 9h-11h e 14h-16h; refinar filtro CNAE'),('Atendimento->Completude','~0.8%','Script inicial pouco persuasivo','Reescrever abertura; max 2 opcoes no menu'),('Lead->Conta Aberta','~28%','Tempo de resposta do consultor','SLA < 5 min; dashboard tempo real'),]
for etapa, taxa, hipotese, acao in gargalos:
    print(f'GARGALO: {etapa} ({taxa})')
    print(f'  Hipotese: {hipotese}')
    print(f'  Acao:     {acao}\n')